Code fine-tune Model 3


## Bước 1: Nạp dữ liệu và kiểm tra môi trường

In [1]:
# Path giúp tạo đường dẫn tệp theo hệ điều hành đang dùng.
from pathlib import Path

# json dùng để đọc từng dòng trong file JSONL.
import json

# PyTorch dùng ở các bước sau để huấn luyện PhoBERT.
import torch

# Lấy thư mục hiện tại mà notebook đang chạy.
project_root = Path.cwd()

# Nếu notebook chạy từ notebooks/, quay lên thư mục gốc dự án.
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

# Tạo đường dẫn tới hai tập dữ liệu đã chốt.
train_path = project_root / "data" / "processed" / "train.json"
dev_path = project_root / "data" / "processed" / "dev.json"

# Hàm đọc file JSON Lines: mỗi dòng là một mẫu dữ liệu riêng.
def load_jsonl(file_path):
    examples = []

    # Mở file bằng UTF-8 để đọc đúng nội dung tiếng Việt.
    with file_path.open("r", encoding="utf-8") as file:
        for line in file:
            # Bỏ qua dòng trống, nếu có.
            if line.strip():
                examples.append(json.loads(line))

    return examples

# Nạp dữ liệu train và dev vào hai danh sách riêng.
train_examples = load_jsonl(train_path)
dev_examples = load_jsonl(dev_path)

# Đếm số câu có đáp án và không có đáp án trong tập train.
train_no_answer = sum(
    example["is_impossible"] for example in train_examples
)
train_has_answer = len(train_examples) - train_no_answer

# Đếm hai nhóm tương tự trong tập dev.
dev_no_answer = sum(
    example["is_impossible"] for example in dev_examples
)
dev_has_answer = len(dev_examples) - dev_no_answer

# In số mẫu để chắc chắn đã nạp đúng dữ liệu.
print("Train:", len(train_examples))
print("  Có đáp án:", train_has_answer)
print("  Không có đáp án:", train_no_answer)

print("Dev:", len(dev_examples))
print("  Có đáp án:", dev_has_answer)
print("  Không có đáp án:", dev_no_answer)

# Kiểm tra notebook đang thấy GPU hay không.
print("CUDA sẵn sàng:", torch.cuda.is_available())

# Nếu có GPU, in tên GPU đang dùng.
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Train: 8000
  Có đáp án: 5417
  Không có đáp án: 2583
Dev: 3814
  Có đáp án: 2653
  Không có đáp án: 1161
CUDA sẵn sàng: False


Mục đích của bước này là xác nhận notebook đọc đúng dữ liệu trước khi huấn luyện. Kết quả mong đợi: train 8.000 mẫu, dev 3.814 mẫu, và máy nhận đúng gpu

-----------------------------------------------------------------------------------------

## Bước 2: Thử tách từ và kiểm tra đáp án
Notebook 06 hiện mới có tiêu đề. Trước khi huấn luyện PhoBERT, ta cần xem underthesea tách câu hỏi/context ra sao và vị trí đáp án thay đổi thế nào. word_tokenize(..., format="text") nối các âm tiết cùng một từ bằng dấu gạch dưới, ví dụ Chàng trai thành Chàng_trai

In [2]:
%pip install underthesea

Note: you may need to restart the kernel to use updated packages.


In [3]:
# word_tokenize dùng underthesea để tách từ tiếng Việt.
from underthesea import word_tokenize

# Chọn một mẫu có đáp án để kiểm tra phép tách từ.
example = next(
    item for item in train_examples
    if not item["is_impossible"]
    and item["answers"]["text"]
    and item["answers"]["answer_start"]
)

# Lấy context, câu hỏi, vị trí bắt đầu và nội dung đáp án gốc.
question = example["question"]
context = example["context"]
answer_text = example["answers"]["text"][0]
answer_start = example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer_text)

# Cắt đáp án ra khỏi context gốc bằng vị trí ký tự ban đầu.
answer_from_context = context[answer_start:answer_end]

# Tách từ; format="text" nối các phần của một từ bằng dấu gạch dưới.
question_segmented = word_tokenize(question, format="text")
context_segmented = word_tokenize(context, format="text")
answer_segmented = word_tokenize(answer_text, format="text")

# Tìm đáp án đã tách trong context đã tách.
segmented_answer_start = context_segmented.find(answer_segmented)

print("Câu hỏi gốc:", question)
print("Câu hỏi sau tách từ:", question_segmented)
print()
print("Đáp án gốc:", answer_text)
print("Đoạn cắt theo answer_start:", answer_from_context)
print("Vị trí đáp án trong context gốc:", answer_start)
print()
print("Đáp án sau tách từ:", answer_segmented)
print("Vị trí đáp án trong context đã tách:", segmented_answer_start)
print("Tìm thấy đáp án đã tách:", segmented_answer_start >= 0)

# Kiểm tra answer_start ban đầu có trỏ đúng đáp án trong context gốc không.
print("answer_start khớp context gốc:", answer_from_context == answer_text)

Câu hỏi gốc: Birmingham phát triển nổi bật về lĩnh vực gì?
Câu hỏi sau tách từ: Birmingham phát_triển nổi_bật về lĩnh_vực gì ?

Đáp án gốc: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Đoạn cắt theo answer_start: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Vị trí đáp án trong context gốc: 179

Đáp án sau tách từ: hoạt_động văn_chương , âm_nhạc , nghệ_thuật và sân_khấu
Vị trí đáp án trong context đã tách: 181
Tìm thấy đáp án đã tách: True
answer_start khớp context gốc: True


Điểm cần quan sát: answer_start là vị trí trong context gốc. Sau khi tách từ, độ dài và vị trí ký tự thay đổi, nên không được dùng nguyên answer_start cũ trên context_segmented. Bước này chỉ giúp ta nhìn rõ thay đổi đó; chưa huấn luyện model.

## Bước 3: Ánh xạ đáp án sau khi tách từ cho toàn bộ dữ liệu

PhoBERT cần câu hỏi và context đã tách từ. Dấu cách bên trong một từ có thể đổi thành dấu gạch dưới, vì vậy answer_start gốc không thể dùng trực tiếp trên context mới. Ta căn chỉnh theo thứ tự ký tự sau khi bỏ khoảng trắng và dấu gạch dưới.

Nếu answer_start gốc sai, chỉ tự sửa khi đáp án có đúng một vị trí trong context. Trường hợp còn mơ hồ sẽ được báo lỗi; không âm thầm bỏ mẫu.


In [4]:
from collections import Counter


def la_phan_tach(character):
    # Khoảng trắng (mọi loại) và dấu gạch dưới do tách từ thêm vào.
    return character.isspace() or character == "_"


def rut_gon(text):
    return "".join(c for c in text if not la_phan_tach(c))


def tim_tat_ca_vi_tri(text, phrase):
    positions = []
    search_from = 0
    while phrase:
        position = text.find(phrase, search_from)
        if position == -1:
            break
        positions.append(position)
        search_from = position + 1
    return positions


_context_cache = {}


def tach_context(context):
    # Trả về (context đã tách, thông tin từng từ).
    # Mỗi từ: (ký tự bắt đầu gốc, ký tự kết thúc gốc, bắt đầu sau tách, kết thúc sau tách).
    if context in _context_cache:
        return _context_cache[context]

    words = word_tokenize(context)
    if isinstance(words, str):
        words = words.split(" ")

    compact = []
    positions = []
    for index, character in enumerate(context):
        if not la_phan_tach(character):
            compact.append(character)
            positions.append(index)
    compact = "".join(compact)

    pointer = 0
    segmented_length = 0
    segmented_words = []
    word_info = []

    for token in words:
        word = token.replace(" ", "_")
        token_compact = rut_gon(token)
        if not token_compact:
            continue

        if compact.startswith(token_compact, pointer):
            found = pointer
        else:
            # Cho phép bỏ qua vài ký tự mà tokenizer đã loại bỏ.
            found = compact.find(token_compact, pointer, pointer + 50)

        if segmented_words:
            segmented_length += 1  # dấu cách giữa hai từ
        seg_start = segmented_length
        segmented_length += len(word)
        segmented_words.append(word)

        if found == -1:
            word_info.append((None, None, seg_start, segmented_length))
            pointer += len(token_compact)
        else:
            original_start = positions[found]
            original_end = positions[found + len(token_compact) - 1] + 1
            word_info.append((original_start, original_end, seg_start, segmented_length))
            pointer = found + len(token_compact)

    result = (" ".join(segmented_words), word_info)
    _context_cache[context] = result
    return result


def anh_xa_span(context, answer_text, answer_start):
    # Trả về (context_tach, dap_an_tach, vi_tri_tach, da_sua_offset, ly_do_loi, mo_rong_theo_tu).
    repaired = False

    def fail(reason):
        return None, None, None, repaired, reason, False

    if context[answer_start:answer_start + len(answer_text)] != answer_text:
        candidates = tim_tat_ca_vi_tri(context, answer_text)
        if not candidates:
            stripped = answer_text.strip()
            candidates = tim_tat_ca_vi_tri(context, stripped)
            answer_text = stripped
        if not candidates:
            return fail("không tìm thấy đáp án trong context gốc")
        # Nếu đáp án lặp, chọn vị trí gần answer_start gốc nhất.
        answer_start = min(candidates, key=lambda p: abs(p - answer_start))
        repaired = True

    answer_end = answer_start + len(answer_text)
    segmented_context, word_info = tach_context(context)

    overlapping = [
        i for i, (o_start, o_end, _, _) in enumerate(word_info)
        if o_start is not None and o_start < answer_end and o_end > answer_start
    ]
    if not overlapping:
        return fail("không có từ nào trùng span đáp án")

    first, last = overlapping[0], overlapping[-1]
    if any(word_info[i][0] is None for i in range(first, last + 1)):
        return fail("từ trong span bị tokenizer đổi ký tự")

    seg_start = word_info[first][2]
    seg_end = word_info[last][3]
    segmented_answer = segmented_context[seg_start:seg_end]

    answer_compact = rut_gon(answer_text)
    segmented_compact = rut_gon(segmented_answer)
    if not answer_compact or answer_compact not in segmented_compact:
        return fail("span sau tách không chứa đáp án gốc")

    expanded = answer_compact != segmented_compact
    return segmented_context, segmented_answer, seg_start, repaired, None, expanded


def xu_ly_tap(examples, bo_mau_loi):
    processed = []
    errors = []
    stats = Counter()

    for example in examples:
        new_example = example.copy()
        new_example["question"] = word_tokenize(example["question"], format="text")
        new_example["context"] = tach_context(example["context"])[0]

        if example["is_impossible"]:
            stats["không có đáp án"] += 1
            processed.append(new_example)
            continue

        answers = example.get("answers", {})
        texts = answers.get("text", [])
        starts = answers.get("answer_start", [])

        if not texts or not starts:
            reason = "thiếu answer_text hoặc answer_start"
            result = None
        else:
            result = anh_xa_span(example["context"], texts[0], starts[0])
            reason = result[4]

        if reason:
            errors.append({"id": example.get("id"), "reason": reason})
            if not bo_mau_loi:
                # Dev: giữ mẫu để chấm điểm theo đáp án gốc, nhãn loss = CLS.
                new_example["answers"] = {"text": [], "answer_start": []}
                processed.append(new_example)
            continue

        _, answer_new, start_new, repaired, _, expanded = result
        stats["ánh xạ được"] += 1
        stats["đã sửa offset"] += int(repaired)
        stats["mở rộng theo ranh giới từ"] += int(expanded)
        new_example["answers"] = {"text": [answer_new], "answer_start": [start_new]}
        processed.append(new_example)

    return processed, errors, stats


train_segmented, train_errors, train_stats = xu_ly_tap(train_examples, bo_mau_loi=True)
dev_segmented, dev_errors, dev_stats = xu_ly_tap(dev_examples, bo_mau_loi=False)

print("Train gốc:", len(train_examples), "| giữ lại:", len(train_segmented), "| loại:", len(train_errors))
print("  Thống kê:", dict(train_stats))
print("  Lý do lỗi:", dict(Counter(e["reason"] for e in train_errors)))
print("Dev gốc:", len(dev_examples), "| giữ lại:", len(dev_segmented), "| mẫu lỗi nhãn:", len(dev_errors))
print("  Thống kê:", dict(dev_stats))
print("  Lý do lỗi:", dict(Counter(e["reason"] for e in dev_errors)))
print("Ví dụ lỗi train:", train_errors[:5])

# Dev phải giữ đủ mẫu để so sánh công bằng giữa các model.
assert len(dev_segmented) == len(dev_examples), "Dev bị mất mẫu"

# Dừng nếu tỉ lệ lỗi bất thường (thường là do tokenizer/phiên bản thư viện).
if len(train_errors) > 0.03 * len(train_examples):
    raise ValueError(
        f"Quá nhiều mẫu train lỗi ({len(train_errors)}); kiểm tra lại: {train_errors[:10]}"
    )

Train gốc: 8000 | giữ lại: 7928 | loại: 72
  Thống kê: {'không có đáp án': 2583, 'ánh xạ được': 5345, 'đã sửa offset': 48, 'mở rộng theo ranh giới từ': 190}
  Lý do lỗi: {'từ trong span bị tokenizer đổi ký tự': 58, 'span sau tách không chứa đáp án gốc': 11, 'không có từ nào trùng span đáp án': 3}
Dev gốc: 3814 | giữ lại: 3814 | mẫu lỗi nhãn: 33
  Thống kê: {'ánh xạ được': 2620, 'đã sửa offset': 13, 'mở rộng theo ranh giới từ': 77, 'không có đáp án': 1161}
  Lý do lỗi: {'span sau tách không chứa đáp án gốc': 12, 'không có từ nào trùng span đáp án': 2, 'từ trong span bị tokenizer đổi ký tự': 19}
Ví dụ lỗi train: [{'id': '0126-0020-0004', 'reason': 'từ trong span bị tokenizer đổi ký tự'}, {'id': '0033-0047-0004', 'reason': 'từ trong span bị tokenizer đổi ký tự'}, {'id': '0021-0037-0004', 'reason': 'từ trong span bị tokenizer đổi ký tự'}, {'id': '0109-0003-0004', 'reason': 'từ trong span bị tokenizer đổi ký tự'}, {'id': '0126-0039-0003', 'reason': 'từ trong span bị tokenizer đổi ký tự'}]


## Bước 4: Nạp tokenizer PhoBERT và xác nhận token ID

PhoBERT cần văn bản đã tách từ. PhoBERT-base-v2 có giới hạn 256 token, vì vậy notebook dùng MAX_LENGTH là 256. Ta cần tokenizer nhanh để lấy offset_mapping. Nếu Transformers chỉ nạp được tokenizer chậm, cell thử dùng tokenizer.json và so sánh token ID với tokenizer chuẩn. Nếu không khớp thì phải dừng, không tạo nhãn sai. [Tài liệu PhoBERT](https://huggingface.co/vinai/phobert-base-v2)


In [5]:
import re

from transformers import AutoTokenizer

CHECKPOINT = "vinai/phobert-base-v2"
MAX_LENGTH = 256
DOC_STRIDE = 64
MAX_QUESTION_TOKENS = 64
SO_MAU_KIEM_TRA = 200

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=False)

CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id
UNK_ID = tokenizer.unk_token_id
assert CLS_ID == 0, "CLS phải ở chỉ số 0 để giải mã no-answer"

_word_cache = {}


def ma_hoa_tu(word):
    # Mỗi từ (đã tách theo khoảng trắng) được BPE độc lập, nên có thể lưu cache.
    if word not in _word_cache:
        ids = tokenizer.encode(word, add_special_tokens=False)
        _word_cache[word] = ids if ids else [UNK_ID]
    return _word_cache[word]


def ma_hoa_van_ban(text):
    # Trả về (ids, từ của từng token, vị trí ký tự [bắt đầu, kết thúc) của từng từ).
    ids = []
    word_of_token = []
    spans = []
    for word_index, match in enumerate(re.finditer(r"\S+", text)):
        spans.append((match.start(), match.end()))
        for token_id in ma_hoa_tu(match.group()):
            ids.append(token_id)
            word_of_token.append(word_index)
    return ids, word_of_token, spans


def ma_hoa_cau_hoi(question):
    ids, _, _ = ma_hoa_van_ban(question)
    return ids[:MAX_QUESTION_TOKENS]


# Kiểm tra cách ghép cặp tự dựng với tokenizer gốc.
samples = [
    (example["question"], example["context"])
    for example in train_segmented[:SO_MAU_KIEM_TRA]
]

mismatch = []
unk_count = 0
total_tokens = 0
for index, (question, context) in enumerate(samples):
    question_ids = ma_hoa_cau_hoi(question)
    if len(question_ids) >= MAX_QUESTION_TOKENS:
        continue  # câu hỏi bị cắt, không so sánh với tokenizer gốc
    context_ids, _, _ = ma_hoa_van_ban(context)
    mine = [CLS_ID] + question_ids + [SEP_ID, SEP_ID] + context_ids + [SEP_ID]
    reference = tokenizer(question, context)["input_ids"]
    unk_count += mine.count(UNK_ID)
    total_tokens += len(mine)
    if mine != reference:
        mismatch.append(index)

print("Số mẫu kiểm tra:", len(samples), "| khác tokenizer gốc:", len(mismatch))
print("Tỉ lệ <unk>: %.3f%%" % (100 * unk_count / max(total_tokens, 1)))
print("MAX_LENGTH:", MAX_LENGTH, "| DOC_STRIDE:", DOC_STRIDE)

if mismatch:
    index = mismatch[0]
    question, context = samples[index]
    question_ids = ma_hoa_cau_hoi(question)
    mine = [CLS_ID] + question_ids + [SEP_ID, SEP_ID] + ma_hoa_van_ban(context)[0] + [SEP_ID]
    reference = tokenizer(question, context)["input_ids"]
    position = next(
        (k for k, (a, b) in enumerate(zip(mine, reference)) if a != b),
        min(len(mine), len(reference)),
    )
    print("Mẫu khác đầu tiên:", index, "| vị trí:", position)
    print("tự dựng:", tokenizer.convert_ids_to_tokens(mine[max(0, position - 6):position + 8]))
    print("gốc    :", tokenizer.convert_ids_to_tokens(reference[max(0, position - 6):position + 8]))
    raise RuntimeError("Cách mã hóa tự dựng khác tokenizer gốc; dừng trước khi tạo nhãn.")

Số mẫu kiểm tra: 200 | khác tokenizer gốc: 0
Tỉ lệ <unk>: 0.105%
MAX_LENGTH: 256 | DOC_STRIDE: 64


## Bước 5: Token hóa, chia cửa sổ và tạo nhãn start/end

Mỗi token context có offset ký tự. Dùng offset đó để đổi span đáp án thành hai chỉ số token: start_positions và end_positions. Context dài được chia thành các cửa sổ chồng lặp. Cửa sổ không chứa đáp án và câu hỏi không đáp án được gán nhãn CLS.


In [6]:
from datasets import Dataset

SO_TOKEN_DAC_BIET = 4  # <s>, </s>, </s>, </s>


def tao_features(examples, co_metadata=False):
    rows = {
        "input_ids": [],
        "attention_mask": [],
        "start_positions": [],
        "end_positions": [],
    }
    if co_metadata:
        rows["example_id"] = []
        rows["offset_mapping"] = []

    for example in examples:
        question_ids = ma_hoa_cau_hoi(example["question"])
        context_ids, word_of_token, spans = ma_hoa_van_ban(example["context"])
        total = len(context_ids)
        window = MAX_LENGTH - SO_TOKEN_DAC_BIET - len(question_ids)
        step = max(1, window - DOC_STRIDE)

        first_token = {}
        last_token = {}
        for token_index, word_index in enumerate(word_of_token):
            first_token.setdefault(word_index, token_index)
            last_token[word_index] = token_index

        # Token đầu/cuối của đáp án theo chỉ số trong context (None nếu không có đáp án).
        answer_tokens = None
        answers = example["answers"]
        if (
            not example["is_impossible"]
            and answers["text"]
            and answers["answer_start"]
        ):
            char_start = answers["answer_start"][0]
            char_end = char_start + len(answers["text"][0])
            hit_words = [
                i for i, (a, b) in enumerate(spans)
                if b > char_start and a < char_end
            ]
            if hit_words:
                answer_tokens = (
                    first_token[hit_words[0]],
                    last_token[hit_words[-1]],
                )

        prefix_length = 1 + len(question_ids) + 2
        window_start = 0
        while True:
            window_end = min(window_start + window, total)

            input_ids = (
                [CLS_ID] + question_ids + [SEP_ID, SEP_ID]
                + context_ids[window_start:window_end] + [SEP_ID]
            )
            padding = MAX_LENGTH - len(input_ids)

            start_position = end_position = 0  # CLS
            if answer_tokens is not None:
                token_start, token_end = answer_tokens
                if token_start >= window_start and token_end < window_end:
                    start_position = prefix_length + token_start - window_start
                    end_position = prefix_length + token_end - window_start

            rows["input_ids"].append(input_ids + [PAD_ID] * padding)
            rows["attention_mask"].append([1] * len(input_ids) + [0] * padding)
            rows["start_positions"].append(start_position)
            rows["end_positions"].append(end_position)

            if co_metadata:
                offsets = (
                    [None] * prefix_length
                    + [list(spans[w]) for w in word_of_token[window_start:window_end]]
                    + [None] * (1 + padding)
                )
                rows["example_id"].append(example["id"])
                rows["offset_mapping"].append(offsets)

            if window_end >= total:
                break
            window_start += step

    return rows


train_features = Dataset.from_dict(tao_features(train_segmented))
dev_features_with_metadata = Dataset.from_dict(
    tao_features(dev_segmented, co_metadata=True)
)

# Metadata dùng để giải mã, không đưa vào đầu vào của model.
dev_features = dev_features_with_metadata.remove_columns(
    ["example_id", "offset_mapping"]
)

train_positive = sum(1 for s in train_features["start_positions"] if s != 0)

print("Train mẫu gốc:", len(train_segmented))
print("Train cửa sổ token:", len(train_features), "| cửa sổ có đáp án:", train_positive)
print("Dev mẫu gốc:", len(dev_segmented))
print("Dev cửa sổ token:", len(dev_features))
assert len(train_features[0]["input_ids"]) == MAX_LENGTH

Train mẫu gốc: 7928
Train cửa sổ token: 8994 | cửa sổ có đáp án: 5444
Dev mẫu gốc: 3814
Dev cửa sổ token: 4179


## Bước 6: Chuẩn bị metrics toàn bộ, answerable và unanswerable

Dự đoán được giải mã về context đã tách từ, sau đó dấu gạch dưới đổi thành khoảng trắng để so sánh với đáp án gốc. Ngoài EM/F1 toàn bộ, notebook tính EM/F1 cho câu có đáp án và accuracy cho câu không có đáp án.


In [7]:
import string
import unicodedata
from collections import Counter, defaultdict

import numpy as np

CLS_INDEX = 0          # <s> luôn ở vị trí 0 với PhoBERT
N_BEST = 20
MAX_ANSWER_TOKENS = 64
NULL_THRESHOLD = 0.0   # giữ cố định giống notebook 04/05

dev_examples_by_id = {}
for original_example, segmented_example in zip(dev_examples, dev_segmented):
    assert original_example["id"] == segmented_example["id"]
    dev_examples_by_id[original_example["id"]] = {
        "context": segmented_example["context"],
        "answers": original_example["answers"],
        "is_impossible": original_example["is_impossible"],
    }

dev_feature_ids = list(dev_features_with_metadata["example_id"])
dev_feature_offsets = list(dev_features_with_metadata["offset_mapping"])


def chuan_hoa(text):
    text = unicodedata.normalize("NFC", text).lower()
    text = "".join(
        c for c in text
        if c not in string.punctuation and not unicodedata.category(c).startswith("P")
    )
    return " ".join(text.split())


def f1_token(prediction_tokens, reference_tokens):
    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def score_prediction(prediction, references):
    prediction_norm = chuan_hoa(prediction)
    if not references:
        # Câu không có đáp án: đúng khi model trả về chuỗi rỗng.
        value = float(prediction_norm == "")
        return value, value

    best_em, best_f1 = 0.0, 0.0
    for reference in references:
        reference_norm = chuan_hoa(reference)
        best_em = max(best_em, float(prediction_norm == reference_norm))
        best_f1 = max(
            best_f1,
            f1_token(prediction_norm.split(), reference_norm.split()),
        )
    return best_em, best_f1


def decode_predictions(start_logits, end_logits, feature_ids, feature_offsets, examples_by_id):
    features_per_example = defaultdict(list)
    for feature_index, example_id in enumerate(feature_ids):
        features_per_example[example_id].append(feature_index)

    predictions = {}
    for example_id, example in examples_by_id.items():
        context = example["context"]
        best_score = -np.inf
        best_text = ""
        null_score = np.inf

        for feature_index in features_per_example.get(example_id, []):
            starts = start_logits[feature_index]
            ends = end_logits[feature_index]
            offsets = feature_offsets[feature_index]

            null_score = min(null_score, starts[CLS_INDEX] + ends[CLS_INDEX])

            start_candidates = np.argsort(starts)[-1:-N_BEST - 1:-1]
            end_candidates = np.argsort(ends)[-1:-N_BEST - 1:-1]

            for start_index in start_candidates:
                for end_index in end_candidates:
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if end_index < start_index:
                        continue
                    if end_index - start_index + 1 > MAX_ANSWER_TOKENS:
                        continue
                    score = starts[start_index] + ends[end_index]
                    if score > best_score:
                        best_score = score
                        best_text = context[
                            offsets[start_index][0]:offsets[end_index][1]
                        ]

        if best_text == "" or null_score > best_score + NULL_THRESHOLD:
            best_text = ""
        predictions[example_id] = best_text

    return predictions


def tinh_metrics(evaluation_prediction):
    start_logits, end_logits = evaluation_prediction.predictions[:2]

    segmented_predictions = decode_predictions(
        start_logits,
        end_logits,
        dev_feature_ids,
        dev_feature_offsets,
        dev_examples_by_id,
    )

    all_em, all_f1 = [], []
    answerable_em, answerable_f1 = [], []
    unanswerable_correct = []

    for example_id, example in dev_examples_by_id.items():
        prediction = segmented_predictions[example_id].replace("_", " ")

        if example["is_impossible"]:
            em, f1 = score_prediction(prediction, [])
            unanswerable_correct.append(em)
        else:
            em, f1 = score_prediction(prediction, example["answers"].get("text", []))
            answerable_em.append(em)
            answerable_f1.append(f1)

        all_em.append(em)
        all_f1.append(f1)

    return {
        "exact_match": 100 * float(np.mean(all_em)),
        "f1": 100 * float(np.mean(all_f1)),
        "answerable_em": 100 * float(np.mean(answerable_em)),
        "answerable_f1": 100 * float(np.mean(answerable_f1)),
        "unanswerable_accuracy": 100 * float(np.mean(unanswerable_correct)),
    }


print("Dev mẫu gốc:", len(dev_examples_by_id))
print("Dev cửa sổ:", len(dev_feature_ids))

Dev mẫu gốc: 3814
Dev cửa sổ: 4179


## Bước 7: Cấu hình fine-tune PhoBERT

Mỗi seed bắt đầu lại từ checkpoint PhoBERT gốc. Batch 4 với tích lũy gradient 8 lần tương đương batch hiệu dụng 32. Model tốt nhất được chọn bằng F1 trên dev. Chỉ chạy sau khi Bước 3 không còn lỗi ánh xạ và Bước 4 xác nhận token ID khớp.


In [8]:
import inspect
import json
import os
import time
import torch

from transformers import (
    AutoModelForQuestionAnswering,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


def save_json(file_path, data):
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with file_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


# ---- Cấu hình có thể chỉnh ----
SEEDS = [42, 123, 1024]
NUM_EPOCHS = 3              # giữ 3 để so sánh công bằng với các model khác
MAX_TRAIN_FEATURES = None   # ví dụ 500 để chạy thử nhanh trên CPU; None = dùng toàn bộ
RESULTS_DIR = project_root / "results" / "finetune" / "phobert"

# ---- Chọn thiết bị ----
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    print("Thiết bị: GPU", torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(os.cpu_count() or 1)
    print("Thiết bị: CPU,", torch.get_num_threads(), "luồng")
    print("CẢNH BÁO: CPU rất chậm; 3 seed x 3 epoch có thể mất nhiều ngày.")
    print("Nên thử trước với MAX_TRAIN_FEATURES nhỏ và NUM_EPOCHS = 1.")
print("BF16:", use_bf16, "| FP16:", use_fp16)

# Tập train dùng khi huấn luyện (có thể rút gọn để chạy thử).
if MAX_TRAIN_FEATURES:
    train_features_used = train_features.select(
        range(min(MAX_TRAIN_FEATURES, len(train_features)))
    )
    print("CHẠY THỬ: chỉ dùng", len(train_features_used), "cửa sổ train.")
else:
    train_features_used = train_features
print("Số cửa sổ train:", len(train_features_used), "| dev:", len(dev_features))


def tao_training_arguments(output_dir, seed):
    arguments = {
        "output_dir": str(output_dir),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": NUM_EPOCHS,
        "weight_decay": 0.01,
        "bf16": use_bf16,
        "fp16": use_fp16,
        "logging_strategy": "steps",
        "logging_steps": 100,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "dataloader_num_workers": 0,
        "dataloader_pin_memory": use_cuda,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed,
    }

    supported = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    # Ép chạy CPU khi không có GPU (tên tham số đổi giữa các phiên bản).
    if not use_cuda:
        if "use_cpu" in supported:
            arguments["use_cpu"] = True
        else:
            arguments["no_cuda"] = True

    strategy_name = (
        "eval_strategy"
        if "eval_strategy" in supported
        else "evaluation_strategy"
    )
    arguments[strategy_name] = "epoch"
    arguments = {
        key: value for key, value in arguments.items()
        if key in supported
    }

    return TrainingArguments(**arguments)

Thiết bị: CPU, 20 luồng
CẢNH BÁO: CPU rất chậm; 3 seed x 3 epoch có thể mất nhiều ngày.
Nên thử trước với MAX_TRAIN_FEATURES nhỏ và NUM_EPOCHS = 1.
BF16: False | FP16: False
Số cửa sổ train: 8994 | dev: 4179


## Bước 8: Huấn luyện ba seed và lưu từng lượt chạy

Vòng lặp tạo model mới cho mỗi seed, train, chọn checkpoint tốt nhất trên dev và lưu metrics cùng dự đoán. Mỗi seed có thư mục riêng nên có thể xem lại kết quả.


In [ ]:
all_seed_results = []

for seed in SEEDS:
    print("\\n" + "=" * 70)
    print("PhoBERT — seed:", seed)
    print("=" * 70)

    set_seed(seed)
    seed_dir = RESULTS_DIR / f"seed-{seed}"
    best_model_dir = seed_dir / "best_model"
    seed_dir.mkdir(parents=True, exist_ok=True)

    metrics_file = seed_dir / "metrics.json"
    if metrics_file.exists() and best_model_dir.exists():
        print("Đã có kết quả seed", seed, "; giữ nguyên.")
        all_seed_results.append(json.loads(metrics_file.read_text(encoding="utf-8")))
        continue

    model = AutoModelForQuestionAnswering.from_pretrained(CHECKPOINT)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer_parameters = inspect.signature(
        Trainer.__init__
    ).parameters
    tokenizer_argument = (
        {"processing_class": tokenizer}
        if "processing_class" in trainer_parameters
        else {"tokenizer": tokenizer}
    )

    trainer = Trainer(
        model=model,
        args=tao_training_arguments(seed_dir, seed),
        train_dataset=train_features_used,
        eval_dataset=dev_features,
        data_collator=data_collator,
        compute_metrics=tinh_metrics,
        **tokenizer_argument,
    )

    start_time = time.time()
    trainer.train()
    training_seconds = time.time() - start_time

    # Trainer đã nạp checkpoint có F1 dev tốt nhất.
    final_metrics = trainer.evaluate()
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(best_model_dir)

    prediction_output = trainer.predict(dev_features)
    start_logits, end_logits = prediction_output.predictions[:2]
    segmented_predictions = decode_predictions(
        start_logits,
        end_logits,
        dev_feature_ids,
        dev_feature_offsets,
        dev_examples_by_id,
    )

    prediction_rows = []
    for example_id, example in dev_examples_by_id.items():
        prediction_rows.append({
            "id": example_id,
            "prediction_text": segmented_predictions[example_id].replace(
                "_", " "
            ),
            "ground_truths": example["answers"].get("text", []),
            "is_impossible": example["is_impossible"],
        })

    result = {
        "model": "phobert",
        "checkpoint": CHECKPOINT,
        "seed": seed,
        "max_length": MAX_LENGTH,
        "doc_stride": DOC_STRIDE,
        "exact_match": float(
            final_metrics.get("eval_exact_match", 0.0)
        ),
        "f1": float(final_metrics.get("eval_f1", 0.0)),
        "answerable_em": float(
            final_metrics.get("eval_answerable_em", 0.0)
        ),
        "answerable_f1": float(
            final_metrics.get("eval_answerable_f1", 0.0)
        ),
        "unanswerable_accuracy": float(
            final_metrics.get("eval_unanswerable_accuracy", 0.0)
        ),
        "training_seconds": round(training_seconds, 2),
        "bf16": use_bf16,
        "device": "cuda" if use_cuda else "cpu",
        "num_epochs": NUM_EPOCHS,
        "train_features_used": len(train_features_used),
    }

    save_json(seed_dir / "metrics.json", result)
    save_json(seed_dir / "predictions.json", prediction_rows)
    all_seed_results.append(result)

    print("EM:", result["exact_match"], "| F1:", result["f1"])
    print("F1 có đáp án:", result["answerable_f1"])
    print("Accuracy không đáp án:", result["unanswerable_accuracy"])
    print("Đã lưu:", seed_dir)

\n======================================================================
PhoBERT — seed: 42


d:\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

## Bước 9: Tính mean +- std và lưu tổng hợp

Mean là điểm trung bình qua ba seed. Độ lệch chuẩn (std) cho biết kết quả dao động nhiều hay ít. Báo cáo seed F1 dev cao nhất riêng, đồng thời dùng mean ± std để thể hiện độ ổn định.


In [ ]:
metric_names = [
    "exact_match",
    "f1",
    "answerable_em",
    "answerable_f1",
    "unanswerable_accuracy",
]

summary = {
    "model": "phobert",
    "checkpoint": CHECKPOINT,
    "seeds": SEEDS,
    "runs": all_seed_results,
    "metrics_mean_std": {},
}

for metric_name in metric_names:
    values = [row[metric_name] for row in all_seed_results]
    summary["metrics_mean_std"][metric_name] = {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
    }

best_run = max(all_seed_results, key=lambda row: row["f1"])
summary["best_seed"] = best_run["seed"]

save_json(RESULTS_DIR / "summary.json", summary)

print("Seed có F1 dev cao nhất:", summary["best_seed"])
for metric_name, values in summary["metrics_mean_std"].items():
    print(
        metric_name,
        f"{values['mean']:.2f} ± {values['std']:.2f}",
    )
print("Đã lưu:", RESULTS_DIR / "summary.json")


## Bước 10: Kiểm tra và ghi kết quả vào báo cáo

Kết quả nằm trong results/finetune/phobert/. Mỗi thư mục seed có best_model, metrics.json và predictions.json; summary.json chứa điểm từng seed, mean ± std và seed tốt nhất.

Trước khi đưa vào báo cáo, xác nhận đủ ba seed, dev vẫn có 3.814 mẫu, metrics không bị thiếu và ghi rõ PhoBERT dùng Underthesea, MAX_LENGTH 256, DOC_STRIDE 64, batch hiệu dụng 32, 3 epoch. Chỉ dùng test.json cho lần đánh giá cuối cùng sau khi đã chốt mô hình bằng dev.
